# Job Recommendation System
This notebook compares three approaches: TF-IDF (Keywords), SBERT (Semantics), and a **Hybrid Model** (The best of both).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('data.csv').fillna('')

# Pre-processing for all models
df['tfidf_features'] = (
    (df['job_title'] + ' ') * 3 + 
    (df['category'] + ' ') * 2 + 
    (df['skills_required'].str.replace(';', ' ') + ' ') * 3 + 
    df['job_description']
)

df['semantic_text'] = (
    "Job Title: " + df['job_title'] + ". " +
    "Category: " + df['category'] + ". " +
    "Skills: " + df['skills_required'].str.replace(';', ', ') + ". " +
    "Description: " + df['job_description']
)

## 1. Setup Models

In [ ]:
# TF-IDF
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['tfidf_features'])

# SBERT
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
sbert_embeddings = sbert_model.encode(df['semantic_text'].tolist(), show_progress_bar=True)

## 2. Hybrid Recommendation Function
The Hybrid model calculates both TF-IDF and SBERT similarities and combines them. 
We use **0.3 for TF-IDF** (to respect keywords) and **0.7 for SBERT** (to understand intent).

In [ ]:
def recommend_hybrid(query, top_n=5, tfidf_weight=0.3, sbert_weight=0.7):
    # 1. TF-IDF Score
    q_tfidf = tfidf_vectorizer.transform([query])
    sims_tfidf = cosine_similarity(q_tfidf, tfidf_matrix).flatten()
    
    # 2. SBERT Score
    q_sbert = sbert_model.encode([query], show_progress_bar=False)
    sims_sbert = cosine_similarity(q_sbert, sbert_embeddings).flatten()
    
    # 3. Combine
    hybrid_sims = (tfidf_weight * sims_tfidf) + (sbert_weight * sims_sbert)
    top_idx = hybrid_sims.argsort()[-top_n:][::-1]
    
    print(f"Hybrid Recommendations for: '{query}'\n")
    for idx in top_idx:
        title = df.iloc[idx]['job_title']
        company = df.iloc[idx]['company_name']
        score = hybrid_sims[idx]
        print(f"- {title} at {company} ({score*100:.2f}% Match)")

recommend_hybrid("Looking for a coding job with Python and SQL in a bank")

## 3. Evaluation (Accuracy)
Let's see how the Hybrid model compares to the individual ones.

In [36]:
def evaluate_all(test_size=50):
    df_train, df_test = train_test_split(df, test_size=test_size, random_state=42)
    
    # Rebuild TF-IDF on train only for fair test
    eval_tfidf_vec = TfidfVectorizer(stop_words='english')
    eval_tfidf_mat = eval_tfidf_vec.fit_transform(df_train['tfidf_features'])
    eval_sbert_mat = sbert_model.encode(df_train['semantic_text'].tolist(), show_progress_bar=False)
    
    hits = {'tfidf': 0, 'sbert': 0, 'hybrid': 0}
    
    print(f"Evaluating {test_size} samples...")
    for _, row in df_test.iterrows():
        query = row['job_description']
        target = row['job_title'].lower()
        
        # TF-IDF
        q_tfidf = eval_tfidf_vec.transform([query])
        s_tfidf = cosine_similarity(q_tfidf, eval_tfidf_mat).flatten()
        
        # SBERT
        q_sbert = sbert_model.encode([query], show_progress_bar=False)
        s_sbert = cosine_similarity(q_sbert, eval_sbert_mat).flatten()
        
        # Hybrid
        s_hybrid = (0.3 * s_tfidf) + (0.7 * s_sbert)
        
        if df_train.iloc[s_tfidf.argsort()[-1]]['job_title'].lower() == target: hits['tfidf'] += 1
        if df_train.iloc[s_sbert.argsort()[-1]]['job_title'].lower() == target: hits['sbert'] += 1
        if df_train.iloc[s_hybrid.argsort()[-1]]['job_title'].lower() == target: hits['hybrid'] += 1
            
    print("\nAccuracy (Top-1 Match):")
    print(f"TF-IDF: {(hits['tfidf']/test_size)*100:.2f}%")
    print(f"SBERT:  {(hits['sbert']/test_size)*100:.2f}%")
    print(f"Hybrid: {(hits['hybrid']/test_size)*100:.2f}%")

evaluate_all(50)

Evaluating 50 samples...

Accuracy (Top-1 Match):
TF-IDF: 94.00%
SBERT:  66.00%
Hybrid: 88.00%
